# Лабораторная работа № 1. Очистка данных

Дрынкин Иван САПР 1.3

**Инструменты:** PySpark, HDFS, YARN, Apache Iceberg.

Цель — подготовить CSV RAWG к разведочному анализу: изучить структуру, выбрать признаки, нормализовать пропуски, привести типы, проверить значения и дубликаты, сохранить результат в Iceberg. Исходный файл не изменяется.

Датасет: https://www.kaggle.com/datasets/atalaydenknalbant/rawg-games-dataset?resource=download

## 1. Импорт библиотек и настройка Spark

In [1]:
import os
from pyspark import SparkConf
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window


def create_spark_configuration(app_name: str) -> SparkConf:
    """Настройка Spark для университетского кластера YARN и Apache Iceberg."""
    user_name = os.getenv("USER", "user0")

    conf = SparkConf()
    conf.setAppName(app_name)
    conf.setMaster("yarn")
    conf.set("spark.submit.deployMode", "client")
    conf.set("spark.executor.memory", "12g")
    conf.set("spark.executor.cores", "8")
    conf.set("spark.executor.instances", "2")
    conf.set("spark.driver.memory", "4g")
    conf.set("spark.driver.cores", "2")
    conf.set(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.0"
    )
    conf.set(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )
    conf.set("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkCatalog")
    conf.set("spark.sql.catalog.spark_catalog.type", "hadoop")
    conf.set(
        "spark.sql.catalog.spark_catalog.warehouse",
        f"hdfs:///user/{user_name}/warehouse"
    )
    conf.set(
        "spark.sql.catalog.spark_catalog.io-impl",
        "org.apache.iceberg.hadoop.HadoopFileIO"
    )
    return conf

### 1.1. Запуск сессии Spark

In [2]:
conf = create_spark_configuration("drynkin RAWG data cleansing")
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
spark

:: loading settings :: url = jar:file:/opt/spark-3.5.2-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/user0/.ivy2/cache
The jars for the packages stored in: /home/user0/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8d7210c9-403b-458b-9841-d72f70f2a856;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.0 in central
:: resolution report :: resolve 703ms :: artifacts dl 25ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spa

## 2. Чтение исходного CSV из HDFS

Spark читает CSV распределённо. На первом этапе все значения оставляются строками, чтобы явно контролировать преобразование типов. Учитываются заголовок, кавычки и экранирование кавычек.

In [3]:
rawg_path = "hdfs:///user/user0/drynkin_directory/rawg/videogames_data.csv"

raw_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .load(rawg_path)
)

print("Количество столбцов в исходном CSV:", len(raw_df.columns))
print("Названия полей:")
print(raw_df.columns)
raw_df.printSchema()

Количество столбцов в исходном CSV: 54
Названия полей:
['id', 'slug', 'name', 'released', 'background_image', 'rating', 'rating_top', 'ratings_count', 'reviews_text_count', 'added', 'metacritic', 'playtime', 'suggestions_count', 'updated', 'reviews_count', 'saturated_color', 'dominant_color', 'platforms', 'stores', 'developers', 'genres', 'tags', 'publishers', 'esrb_rating', 'added_by_status', 'metacritic_url', 'ratings', 'clip', 'name_original', 'reddit_url', 'movies_count', 'reactions', 'background_image_additional', 'website', 'reddit_count', 'achievements_count', 'youtube_count', 'short_screenshots', 'creators_count', 'twitch_count', 'alternative_names', 'parent_achievements_count', 'additions_count', 'metacritic_platforms', 'screenshots_count', 'parent_platforms', 'game_series_count', 'description', 'description_raw', 'parents_count', 'reddit_name', 'reddit_description', 'reddit_logo', 'tba']
root
 |-- id: string (nullable = true)
 |-- slug: string (nullable = true)
 |-- name: str

### 2.1. Первичный просмотр строк

In [4]:
raw_df.select("id", "name", "released", "rating", "ratings_count", "metacritic", "genres").show(
    10, truncate=70
)

+-----+--------------------------------+----------+------+-------------+----------+--------------+
|   id|                            name|  released|rating|ratings_count|metacritic|        genres|
+-----+--------------------------------+----------+------+-------------+----------+--------------+
| 3498|              Grand Theft Auto V|2013-09-17|  4.47|       7409.0|      92.0|        Action|
| 3328|        The Witcher 3: Wild Hunt|2015-05-18|  4.64|       7234.0|      92.0|    Action|RPG|
| 4200|                        Portal 2|2011-04-18|  4.58|       6118.0|      95.0|Shooter|Puzzle|
| 4291|Counter-Strike: Global Offensive|2012-08-21|  3.57|       3638.0|      81.0|       Shooter|
| 5286|                     Tomb Raider|2013-03-05|  4.06|       4103.0|      86.0|        Action|
|13536|                          Portal|2007-10-09|  4.49|       5030.0|      90.0| Action|Puzzle|
|12020|                   Left 4 Dead 2|2009-11-17|   4.1|       3408.0|      89.0|Action|Shooter|
|   28|   

## 3. Выбор признаков

Из 54 исходных столбцов оставляем идентификаторы, даты, рейтинги, счётчики популярности, жанры, платформы, магазины, разработчиков, издателей и вспомогательные категории. Большие описания и URL для текущего анализа не нужны. Списки категорий, разделённые `|`, сохраняются до разведочного анализа.

In [5]:
selected_columns = [
    "id", "slug", "name", "released", "rating", "rating_top",
    "ratings_count", "reviews_text_count", "added", "metacritic",
    "playtime", "suggestions_count", "updated", "reviews_count",
    "platforms", "stores", "developers", "genres", "tags", "publishers",
    "esrb_rating", "movies_count", "reddit_count", "achievements_count",
    "youtube_count", "twitch_count", "parent_achievements_count",
    "additions_count", "screenshots_count", "parent_platforms",
    "game_series_count", "parents_count", "tba"
]

missing_columns = [column for column in selected_columns if column not in raw_df.columns]
if missing_columns:
    raise ValueError(f"В исходном CSV отсутствуют ожидаемые поля: {missing_columns}")

df = raw_df.select(*selected_columns)
print("Количество выбранных столбцов:", len(df.columns))
df.show(5, truncate=60)

Количество выбранных столбцов: 33


26/10/29 23:48:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----+-------------------------------+--------------------------------+----------+------+----------+-------------+------------------+-------+----------+--------+-----------------+-------------------+-------------+------------------------------------------------------------+------------------------------------------------------------+----------------------------------------+--------------+------------------------------------------------------------+---------------------+--------------------------------------------------------+------------+------------+------------------+-------------+------------+-------------------------+---------------+-----------------+--------------------------------------------+-----------------+-------------+----+
|  id|                           slug|                            name|  released|rating|rating_top|ratings_count|reviews_text_count|  added|metacritic|playtime|suggestions_count|            updated|reviews_count|                                         

## 4. Нормализация маркеров пропущенных значений

Пустые строки и маркеры `nan`, `null`, `none`, `na`, `n/a` преобразуются в настоящий `NULL` до приведения типов.

In [6]:
null_markers = ["", "nan", "null", "none", "na", "n/a"]

normalized_exprs = []
for column in df.columns:
    trimmed = F.trim(F.col(column))
    normalized_exprs.append(
        F.when(F.lower(trimmed).isin(null_markers), F.lit(None).cast("string"))
         .otherwise(trimmed)
         .alias(column)
    )

df = df.select(*normalized_exprs)
print("Пример после нормализации маркеров:")
df.select("id", "name", "released", "rating", "genres", "developers").show(5, truncate=70)

Пример после нормализации маркеров:


+----+--------------------------------+----------+------+--------------+----------------------------------------+
|  id|                            name|  released|rating|        genres|                              developers|
+----+--------------------------------+----------+------+--------------+----------------------------------------+
|3498|              Grand Theft Auto V|2013-09-17|  4.47|        Action|           Rockstar North|Rockstar Games|
|3328|        The Witcher 3: Wild Hunt|2015-05-18|  4.64|    Action|RPG|                          CD PROJEKT RED|
|4200|                        Portal 2|2011-04-18|  4.58|Shooter|Puzzle|                          Valve Software|
|4291|Counter-Strike: Global Offensive|2012-08-21|  3.57|       Shooter|Valve Software|Hidden Path Entertainment|
|5286|                     Tomb Raider|2013-03-05|  4.06|        Action|                        Crystal Dynamics|
+----+--------------------------------+----------+------+--------------+----------------

## 5. Преобразование типов

Счётчики приводятся к целому типу, оценки и `playtime` — к числовому, `released` — к дате, `updated` — к timestamp, `tba` — к boolean. Строки вида `7409.0` сначала преобразуются в число с плавающей точкой, затем в целое.

In [7]:
double_columns = ["rating", "rating_top", "playtime"]
count_columns = [
    "ratings_count", "reviews_text_count", "added", "suggestions_count",
    "reviews_count", "movies_count", "reddit_count", "achievements_count",
    "youtube_count", "twitch_count", "parent_achievements_count",
    "additions_count", "screenshots_count", "game_series_count", "parents_count"
]

for column in double_columns:
    df = df.withColumn(column, F.col(column).cast("double"))

for column in count_columns:
    df = df.withColumn(column, F.col(column).cast("double").cast("long"))

df = df.withColumn("id", F.col("id").cast("double").cast("long"))
df = df.withColumn("metacritic", F.col("metacritic").cast("double").cast("long"))
df = df.withColumn("released", F.to_date(F.col("released"), "yyyy-MM-dd"))
df = df.withColumn("updated", F.to_timestamp(F.col("updated"), "yyyy-MM-dd'T'HH:mm:ss"))

# В esrb_rating исходник содержит JSON-подобную строку; извлекаем название возрастного рейтинга.
esrb_name = F.regexp_extract(F.col("esrb_rating"), r'"name"\s*:\s*"([^"]+)"', 1)
df = df.withColumn(
    "esrb_rating",
    F.when(esrb_name != "", esrb_name).otherwise(F.col("esrb_rating"))
)

tba_value = F.lower(F.col("tba"))
df = df.withColumn(
    "tba",
    F.when(tba_value.isin("true", "1", "yes"), F.lit(True))
     .when(tba_value.isin("false", "0", "no"), F.lit(False))
     .otherwise(F.lit(None).cast("boolean"))
)

df.printSchema()
df.select(
    "id", "name", "released", "rating", "rating_top", "ratings_count",
    "metacritic", "playtime", "updated", "tba", "esrb_rating"
).show(10, truncate=60)

root
 |-- id: long (nullable = true)
 |-- slug: string (nullable = true)
 |-- name: string (nullable = true)
 |-- released: date (nullable = true)
 |-- rating: double (nullable = true)
 |-- rating_top: double (nullable = true)
 |-- ratings_count: long (nullable = true)
 |-- reviews_text_count: long (nullable = true)
 |-- added: long (nullable = true)
 |-- metacritic: long (nullable = true)
 |-- playtime: double (nullable = true)
 |-- suggestions_count: long (nullable = true)
 |-- updated: timestamp (nullable = true)
 |-- reviews_count: long (nullable = true)
 |-- platforms: string (nullable = true)
 |-- stores: string (nullable = true)
 |-- developers: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- publishers: string (nullable = true)
 |-- esrb_rating: string (nullable = true)
 |-- movies_count: long (nullable = true)
 |-- reddit_count: long (nullable = true)
 |-- achievements_count: long (nullable = true)
 |-- youtube_count: lon

+-----+--------------------------------+----------+------+----------+-------------+----------+--------+-------------------+----+------------+
|   id|                            name|  released|rating|rating_top|ratings_count|metacritic|playtime|            updated| tba| esrb_rating|
+-----+--------------------------------+----------+------+----------+-------------+----------+--------+-------------------+----+------------+
| 3498|              Grand Theft Auto V|2013-09-17|  4.47|       5.0|         7409|        92|    74.0|2026-06-27 07:31:06|NULL|      Mature|
| 3328|        The Witcher 3: Wild Hunt|2015-05-18|  4.64|       5.0|         7234|        92|    43.0|2026-06-27 08:52:50|NULL|      Mature|
| 4200|                        Portal 2|2011-04-18|  4.58|       5.0|         6118|        95|    11.0|2026-06-27 08:52:52|NULL|Everyone 10+|
| 4291|Counter-Strike: Global Offensive|2012-08-21|  3.57|       4.0|         3638|        81|    64.0|2026-06-24 18:04:28|NULL|      Mature|
| 5286

### 5.1. Исправление недопустимых числовых значений

`rating` должен находиться в диапазоне 0–5, `metacritic` — 0–100; отрицательные счётчики и `playtime` некорректны. Такие значения заменяются на `NULL`, а не на ноль.

In [8]:
df = df.withColumn(
    "rating",
    F.when(F.col("rating").between(0, 5), F.col("rating")).otherwise(F.lit(None).cast("double"))
)
df = df.withColumn(
    "rating_top",
    F.when(F.col("rating_top").between(0, 5), F.col("rating_top")).otherwise(F.lit(None).cast("double"))
)
df = df.withColumn(
    "metacritic",
    F.when(F.col("metacritic").between(0, 100), F.col("metacritic")).otherwise(F.lit(None).cast("long"))
)

for column in count_columns + ["playtime"]:
    df = df.withColumn(
        column,
        F.when(F.col(column) >= 0, F.col(column)).otherwise(F.lit(None))
    )

## 6. Обработка обязательных полей и категорий

Удаляем строки без `id` или `name`. Пропуски категорий маркируются как `Unknown`; числовые пропуски сохраняются как `NULL`, чтобы не искажать статистику.

In [9]:
rows_before_key_filter = df.count()
df = df.filter(F.col("id").isNotNull() & F.col("name").isNotNull())
rows_after_key_filter = df.count()

print(f"Строк до фильтрации обязательных полей: {rows_before_key_filter:,}")
print(f"Строк после фильтрации обязательных полей: {rows_after_key_filter:,}")
print(f"Удалено строк без id или name: {rows_before_key_filter - rows_after_key_filter:,}")

categorical_columns = [
    "slug", "platforms", "stores", "developers", "genres", "tags",
    "publishers", "esrb_rating", "parent_platforms"
]
df = df.fillna("Unknown", subset=categorical_columns)

Строк до фильтрации обязательных полей: 899,585
Строк после фильтрации обязательных полей: 899,545
Удалено строк без id или name: 40


## 7. Поиск повторяющихся идентификаторов

Для повторов `id` оставляем запись с самой поздней датой `updated`; при равенстве дат порядок стабилизируется по названию.

In [10]:
duplicate_ids_before = (
    df.groupBy("id").count()
      .filter(F.col("count") > 1)
      .orderBy(F.desc("count"), F.asc("id"))
)
print("Количество id с повторениями:", duplicate_ids_before.count())
duplicate_ids_before.show(20, truncate=False)

rows_before_deduplication = df.count()
window_by_id = Window.partitionBy("id").orderBy(
    F.col("updated").desc_nulls_last(),
    F.col("name").asc_nulls_last()
)
df = (
    df.withColumn("_row_number", F.row_number().over(window_by_id))
      .filter(F.col("_row_number") == 1)
      .drop("_row_number")
)
rows_after_deduplication = df.count()
print(f"Строк до удаления дубликатов: {rows_before_deduplication:,}")
print(f"Строк после удаления дубликатов: {rows_after_deduplication:,}")
print(f"Удалено повторных строк: {rows_before_deduplication - rows_after_deduplication:,}")

Количество id с повторениями: 0


+---+-----+
|id |count|
+---+-----+
+---+-----+



Строк до удаления дубликатов: 899,545
Строк после удаления дубликатов: 899,545
Удалено повторных строк: 0


## 8. Контроль качества очищенного набора

После преобразований пересчитываем пропуски. Для категорий `Unknown` считается отсутствующей информацией.

In [11]:
missing_after_exprs = []
for column in df.columns:
    condition = F.col(column).isNull()
    if column in categorical_columns:
        condition = condition | (F.col(column) == "Unknown")
    missing_after_exprs.append(F.sum(F.when(condition, 1).otherwise(0)).alias(column))

missing_after = df.agg(*missing_after_exprs).first().asDict()
final_row_count = df.count()
missing_report = [
    (column, int(missing_after[column] or 0),
     round(100 * int(missing_after[column] or 0) / final_row_count, 2) if final_row_count else 0.0)
    for column in df.columns
]
spark.createDataFrame(
    missing_report,
    ["column", "missing_count", "missing_percent"]
).orderBy(F.desc("missing_percent"), F.desc("missing_count")).show(40, truncate=False)

+-------------------------+-------------+---------------+
|column                   |missing_count|missing_percent|
+-------------------------+-------------+---------------+
|tba                      |899545       |100.0          |
|additions_count          |897739       |99.8           |
|reddit_count             |897661       |99.79          |
|parents_count            |896179       |99.63          |
|metacritic               |892415       |99.21          |
|twitch_count             |889691       |98.9           |
|reviews_text_count       |889140       |98.84          |
|game_series_count        |888729       |98.8           |
|youtube_count            |888340       |98.75          |
|movies_count             |886653       |98.57          |
|rating                   |880870       |97.92          |
|rating_top               |878793       |97.69          |
|parent_achievements_count|870373       |96.76          |
|achievements_count       |870350       |96.75          |
|playtime     

### 8.1. Итоговая схема и основные показатели

Проверяем типы ключевых полей и несколько строк перед записью в Iceberg.

In [12]:
print(f"Итоговое количество игр: {final_row_count:,}")
print(f"Количество столбцов: {len(df.columns)}")
df.printSchema()
df.select(
    "id", "name", "released", "rating", "ratings_count", "metacritic",
    "playtime", "genres", "parent_platforms", "developers"
).show(10, truncate=65)

Итоговое количество игр: 899,545
Количество столбцов: 33
root
 |-- id: long (nullable = true)
 |-- slug: string (nullable = false)
 |-- name: string (nullable = true)
 |-- released: date (nullable = true)
 |-- rating: double (nullable = true)
 |-- rating_top: double (nullable = true)
 |-- ratings_count: long (nullable = true)
 |-- reviews_text_count: long (nullable = true)
 |-- added: long (nullable = true)
 |-- metacritic: long (nullable = true)
 |-- playtime: double (nullable = true)
 |-- suggestions_count: long (nullable = true)
 |-- updated: timestamp (nullable = true)
 |-- reviews_count: long (nullable = true)
 |-- platforms: string (nullable = false)
 |-- stores: string (nullable = false)
 |-- developers: string (nullable = false)
 |-- genres: string (nullable = false)
 |-- tags: string (nullable = false)
 |-- publishers: string (nullable = false)
 |-- esrb_rating: string (nullable = false)
 |-- movies_count: long (nullable = true)
 |-- reddit_count: long (nullable = true)
 |-- a

+---+-------------------------------------+----------+------+-------------+----------+--------+---------------------------------------------+--------------------------------------------------------------+-----------------------------------------------------------------+
| id|                                 name|  released|rating|ratings_count|metacritic|playtime|                                       genres|                                              parent_platforms|                                                       developers|
+---+-------------------------------------+----------+------+-------------+----------+--------+---------------------------------------------+--------------------------------------------------------------+-----------------------------------------------------------------+
|  7|                            Laserlife|2015-09-22|  2.89|            8|      NULL|     5.0|                                 Action|Indie|                           PC|PlayStation|Xbox

## 9. Сохранение результата в Apache Iceberg

Создаём личную базу и таблицу. При повторном запуске пересоздаётся только таблица `drynkin`; исходный CSV остаётся неизменным.

In [13]:
database_name = "drynkin_database"
table_name = "drynkin_lab1_table"
full_table_name = f"spark_catalog.{database_name}.{table_name}"

spark.sql(f"CREATE DATABASE IF NOT EXISTS spark_catalog.{database_name}")
spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")

df.writeTo(full_table_name).using("iceberg").create()
print(f"Сохранена таблица: {full_table_name}")
spark.sql(f"DESCRIBE TABLE {full_table_name}").show(truncate=False)

Сохранена таблица: spark_catalog.drynkin_database.drynkin_lab1_table
+------------------+---------+-------+
|col_name          |data_type|comment|
+------------------+---------+-------+
|id                |bigint   |NULL   |
|slug              |string   |NULL   |
|name              |string   |NULL   |
|released          |date     |NULL   |
|rating            |double   |NULL   |
|rating_top        |double   |NULL   |
|ratings_count     |bigint   |NULL   |
|reviews_text_count|bigint   |NULL   |
|added             |bigint   |NULL   |
|metacritic        |bigint   |NULL   |
|playtime          |double   |NULL   |
|suggestions_count |bigint   |NULL   |
|updated           |timestamp|NULL   |
|reviews_count     |bigint   |NULL   |
|platforms         |string   |NULL   |
|stores            |string   |NULL   |
|developers        |string   |NULL   |
|genres            |string   |NULL   |
|tags              |string   |NULL   |
|publishers        |string   |NULL   |
+------------------+---------+----

### 9.1. Контрольная проверка сохранённой таблицы

In [14]:
saved_df = spark.table(full_table_name)
print(f"Количество строк в Iceberg: {saved_df.count():,}")
print("Количество столбцов в Iceberg:", len(saved_df.columns))
saved_df.select(
    "id", "name", "released", "rating", "ratings_count", "metacritic", "playtime", "genres"
).show(10, truncate=70)

Количество строк в Iceberg: 899,545
Количество столбцов в Iceberg: 33


+---+-------------------------------------+----------+------+-------------+----------+--------+---------------------------------------------+
| id|                                 name|  released|rating|ratings_count|metacritic|playtime|                                       genres|
+---+-------------------------------------+----------+------+-------------+----------+--------+---------------------------------------------+
|  7|                            Laserlife|2015-09-22|  2.89|            8|      NULL|     5.0|                                 Action|Indie|
| 22|                             Farpoint|2017-05-16|  3.49|           88|        71|    NULL|                                      Shooter|
| 26|                          Nex Machina|2017-06-19|   3.9|          153|        89|     3.0|                                 Action|Indie|
| 29|  South Park: The Fractured But Whole|2017-03-31|  4.04|          819|        80|    12.0|                                Adventure|RPG|
| 54| 

## 10. Выводы по очистке

В ходе подготовки датасета RAWG были выполнены выбор признаков, нормализация маркеров пропусков, преобразование типов данных и проверка допустимых диапазонов числовых значений.

Обработка обязательных полей: до фильтрации было 899 585 строк. После удаления записей, в которых отсутствовал идентификатор id или название игры name, осталось 899 545 строк, исключено 40 записей.

Проверка повторяющихся идентификаторов: повторяющихся значений id не обнаружено, до проверки дубликатов и после неё количество строк составило 899 545, на этапе удаления повторов по идентификатору исключений строк не было.

Сохранение результата: подготовленный набор данных сохранён в Apache Iceberg в таблицу spark_catalog.drynkin_database.drynkin_lab1_table, которая используется в следующем ноутбуке для разведочного анализа.

В результате была подготовлена очищенная версия набора данных для дальнейшего исследования характеристик видеоигр.


In [15]:
# Освобождаем ресурсы YARN после завершения очистки.
spark.stop()